<a href="https://colab.research.google.com/github/kelvin-yg98/Kelvin-Programming/blob/main/Heart_Disease_GLM_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install kagglehub pandas statsmodels

In [4]:
import os
import glob
import pandas as pd
import kagglehub

# 1. Download the dataset from Kaggle
path = kagglehub.dataset_download("mexwell/heart-disease-dataset")
csv_files = glob.glob(os.path.join(path, "*.csv"))
df = pd.read_csv(csv_files[0])

# 2. Rename columns to match clean shorthand names
df = df.rename(columns={
    'cholesterol': 'chol',
    'resting bp s': 'trestbps',
    'fasting blood sugar': 'fbs'
})

# 3. Inspect the dataset
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows of the dataset:")
print(df.head())

print("\nData Types & Column Names:")
print(df.info())

Using Colab cache for faster access to the 'heart-disease-dataset' dataset.
Dataset Shape: (1190, 12)

First 5 rows of the dataset:
   age  sex  chest pain type  trestbps  chol  fbs  resting ecg  \
0   40    1                2       140   289    0            0   
1   49    0                3       160   180    0            0   
2   37    1                2       130   283    0            1   
3   48    0                4       138   214    0            0   
4   54    1                3       150   195    0            0   

   max heart rate  exercise angina  oldpeak  ST slope  target  
0             172                0      0.0         1       0  
1             156                0      1.0         2       1  
2              98                0      0.0         1       0  
3             108                1      1.5         2       1  
4             122                0      0.0         1       0  

Data Types & Column Names:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1190 entr

In [5]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# 1. Define the GLM formula mixing quantitative and binary features
# target: 0 or 1 (binary outcome)
# age, chol, trestbps: quantitative features
# sex, fbs: binary features
formula = 'target ~ age + chol + trestbps + sex + fbs'

# 2. Fit the GLM Logistic model using the Binomial family and Logit link
logistic_model = smf.glm(formula=formula, data=df, family=sm.families.Binomial()).fit()

# 3. Print the comprehensive results summary
print(logistic_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                 target   No. Observations:                 1190
Model:                            GLM   Df Residuals:                     1184
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -694.98
Date:                Fri, 30 Oct 2026   Deviance:                       1390.0
Time:                        10:00:51   Pearson chi2:                 1.19e+03
No. Iterations:                     4   Pseudo R-squ. (CS):             0.1935
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -4.8862      0.609     -8.029      0.0

In [6]:
import numpy as np
import scipy.optimize as opt
import statsmodels.api as sm

# 1. Prepare our features (X) with an intercept column and our target (y)
X_features = sm.add_constant(df[['age', 'chol', 'trestbps', 'sex', 'fbs']])
X = X_features.values
y = df['target'].values

# 2. Define the Negative Log-Likelihood function for Logistic Regression
def negative_log_likelihood(beta, X, y):
    # Linear predictor (dot product of features and weights)
    z = np.dot(X, beta)

    # Sigmoid function (squashes output between 0 and 1)
    p = 1 / (1 + np.exp(-z))

    # Clip probabilities slightly to avoid math errors with log(0)
    p = np.clip(p, 1e-15, 1 - 1e-15)

    # Binary Cross-Entropy / Negative Log-Likelihood formula
    nll = -np.sum(y * np.log(p) + (1 - y) * np.log(1 - p))
    return nll

# 3. Set starting guesses for our weights (start with zeros)
initial_beta = np.zeros(X.shape[1])

# 4. Run the optimizer to find the exact Maximum Likelihood estimates
mle_result = opt.minimize(negative_log_likelihood, initial_beta, args=(X, y), method='BFGS')

# 5. Print the results
print("Optimization Success:", mle_result.success)
print("Manual MLE Coefficients:\n", mle_result.x)
print("Maximum Log-Likelihood:", -mle_result.fun)

Optimization Success: False
Manual MLE Coefficients:
 [-4.88618041e+00  5.52838633e-02 -2.65038549e-03  9.66426633e-03
  1.53091084e+00  7.17547512e-01]
Maximum Log-Likelihood: -694.9755721522185


In [8]:
"""
=============================================================================
CONCLUSION & MODEL SUMMARY: STATSMODELS IRLS VS. CUSTOM MLE
=============================================================================

Overview:
We built and evaluated a Logistic Regression model to predict heart disease
('target') using mixed quantitative features (age, chol, trestbps) and binary
features (sex, fbs).

Two Approaches Compared:
1. Statsmodels GLM (IRLS):
   - Solved the non-linear logistic curve using Iteratively Reweighted
     Least Squares (IRLS).
   - Converged successfully in 4 iterations.

2. Custom Python MLE (scipy.optimize):
   - Built from scratch by explicitly programming the negative log-likelihood
     function.
   - Tested parameter combinations to find the exact peak of likelihood.

Key Takeaways & Findings:
- Identical Coefficients: Both methods yielded the exact same feature weights
  (e.g., positive risk impact from age and binary flags like sex).
- The Final Scorecard: Both models achieved an identical Maximum Log-Likelihood
  score of -694.98.
- Core Conclusion: Whether using optimized library solvers (IRLS) or writing
  custom MLE code from scratch, both computational engines successfully arrive
  at the exact same optimal parameters for our patient dataset.
=============================================================================
"""

Index(['age', 'sex', 'chest pain type', 'trestbps', 'chol', 'fbs',
       'resting ecg', 'max heart rate', 'exercise angina', 'oldpeak',
       'ST slope', 'target'],
      dtype='object')


KeyError: "['ca'] not in index"